In [1]:
# --- LOAD PROJECTION WEIGHTS ---
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import torch.nn as nn
import os


C_max = 512
C_out = 256

PROJ_PATH = "/home/chen_le/openset_detection/scripts/projection.pt"
assert os.path.isfile(PROJ_PATH), f"Projection file not found: {PROJ_PATH}"

# create the layer
proj = nn.Linear(C_max, C_out).to(device)
proj.eval()

# load state
ckpt = torch.load(PROJ_PATH, map_location=device)
state = ckpt.get("state_dict", ckpt)
proj.load_state_dict(state, strict=True)

# checksum
with torch.no_grad():
    wsum = torch.cat([p.flatten().detach().cpu() for p in proj.parameters()]).sum().item()
print(f"[INFO] Loaded projection from {PROJ_PATH} (checksum: {wsum:.6f})")

[INFO] Loaded projection from /home/chen_le/openset_detection/scripts/projection.pt (checksum: 1.197872)


In [2]:
import os
import cv2
import xml.etree.ElementTree as ET

def load_image_and_annotations(image_path: str):
    # Construct annotation path (Pascal VOC format)
    base_dir = os.path.dirname(os.path.dirname(image_path))
    annotations_dir = os.path.join(base_dir, "Annotations")
    xml_filename = os.path.splitext(os.path.basename(image_path))[0] + ".xml"
    xml_path = os.path.join(annotations_dir, xml_filename)

    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # Parse XML
    detections = []
    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        bndbox = obj.find("bndbox")
        xmin = int(float(bndbox.find("xmin").text))
        ymin = int(float(bndbox.find("ymin").text))
        xmax = int(float(bndbox.find("xmax").text))
        ymax = int(float(bndbox.find("ymax").text))

        detections.append({
            "obj_name": name,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })

    return img, detections


In [3]:
from outlier_rejector import (
    load_and_prepare_model,
    precompute_maha_state,
    reject_outlier_detections)

# Outlier Rejection
_path = '/home/chen_le/openset_detection/scripts/YOLOv8/training/runs/detect'
_model, _hooks = load_and_prepare_model(_path + '/train_lru1/weights/best.pt')

yolov8_detector =  _model
yolov8_hooks = _hooks

_path = '/home/chen_le/openset_detection/results/YOLOv8/mahalanobis/custom'
ood_paths_dict = {
    'feature_id_train'  : f'{_path}/train/frcnn_GMMDet_Voc_lru1_yolo_feature_id_train.npy',
    'train_labels'      : f'{_path}/train/frcnn_GMMDet_Voc_lru1_yolo_train_labels.npy',
    'feature_id_val'    : f'{_path}/val/frcnn_GMMDet_Voc_lru1_yolo_feature_id_val.npy',
    'val_labels'        : f'{_path}/val/frcnn_GMMDet_Voc_lru1_yolo_val_labels.npy',
    'feature_test'      : f'{_path}/testOOD/frcnn_GMMDet_Voc_lru1_yolo_feature_ood.npy',
    'test_labels'       : f'{_path}/testOOD/frcnn_GMMDet_Voc_lru1_yolo_test_labels.npy',
}

maha_state = precompute_maha_state(ood_paths_dict, tpr_target=0.95)

Class Thresholds:
    drone : -116.74044799804688
    lander: -13.173871040344238
    lru2  : -31.193859100341797


In [4]:
# drone, lander, lru2, background
base_path = "/volume/hot_storage/slurm_data/chen_le/ARCHES"
image_paths = [
    f"{base_path}/lru1_all/JPEGImages/lru_1657025857_914.jpg",
    f"{base_path}/lru1_all/JPEGImages/lru_1656862394_253.jpg",
    f"{base_path}/lru1_all/JPEGImages/lru_1656862489_754.jpg",
    f"{base_path}/lru1_all/JPEGImages/lru_1657024025_714.jpg"
]

# Outlier Rejection
for image_path in image_paths:
    img, detections = load_image_and_annotations(image_path)
    detections, outliers = reject_outlier_detections(detections,
                                                     img,
                                                     yolov8_detector,
                                                     yolov8_hooks,
                                                     maha_state,
                                                     image_path)
    print(detections)

[{'obj_name': 'drone', 'xmin': 148, 'ymin': 1, 'xmax': 234, 'ymax': 34}]
[{'obj_name': 'lander', 'xmin': 1, 'ymin': 1, 'xmax': 757, 'ymax': 913}]
[{'obj_name': 'lru2', 'xmin': 641, 'ymin': 109, 'xmax': 794, 'ymax': 305}]
[]


/home/chen_le/miniforge3/envs/tmnf_os/lib/python3.7/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /pytorch/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


In [5]:
from pathlib import Path
from tqdm.auto import tqdm
import os

# test.txt
txt_file = "/volume/hot_storage/slurm_data/chen_le/ARCHES/lru1_all/ImageSets/YOLO/test.txt"
image_paths = [line.strip()
               for line in Path(txt_file).read_text().splitlines()
               if line.strip() and not line.lstrip().startswith("#")]

# Outlier Rejection with progress bar
all_outliers = []
pbar = tqdm(image_paths, desc="Outlier check", unit="img")
for image_path in pbar:
    img, detections = load_image_and_annotations(image_path)
    detections, outliers = reject_outlier_detections(
        detections, img, yolov8_detector, yolov8_hooks, maha_state, image_path
    )
    if outliers:  # only if non-empty list
        all_outliers.append(outliers)

    # show quick status on the bar
    pbar.set_postfix(last=os.path.basename(image_path), outliers=len(outliers))

# Summary (last image's outliers printed to mirror your original)
print(all_outliers[-1] if all_outliers else [])
print(f"Processed {len(image_paths)} images; total outlier boxes: {sum(len(x) for x in all_outliers)}")


Outlier check:   0%|          | 0/2720 [00:00<?, ?img/s]

[]
Processed 2720 images; total outlier boxes: 48


In [8]:
import pprint

pprint.pprint(all_outliers)

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [{'obj_name': 'lander',
   'path': '/volume/hot_storage/slurm_data/chen_le/ARCHES/lru1_all/JPEGImages/lru_1657026290_614.jpg',
   'xmax': 1257,
   'xmin': 1166,
   'ymax': 105,
   'ymin': 1}],
 [],
 [],
 [],
 [{'obj_name': 'lander',
   'path': '/volume/hot_storage/slurm_data/chen_le/ARCHES/lru1_all/JPEGImages/lru_1657025846_414.jpg',
   'xmax': 1292,
   'xmin': 973,
   'ymax': 166,
   'ymin': 1}],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [{'obj_name': 'lru2',
   'path': '/volume/hot_storage/slurm_data/chen_le/ARCHES/lru1_all/JPEGImages/lru_1657026087_714.jpg',
   'xmax': 1292,
   'xmin': 1235,
   'ymax': 44,
   'ymin': 1},
  {'obj_name': 'lander',
   'path': '/volume/hot_storage/slurm_data/chen_le/ARCHES/lru1_all/JPEGImages/lru_1657026087_714.jpg',
   'xmax': 1292,
   'xmin': 1256,
   'ymax': 58,
   'ymin': 

In [14]:
# --- Generate & save a fixed projection (run once) ---
import os
import torch
import torch.nn as nn

# Must match your feature pipeline
C_MAX = 512
C_OUT = 256

# Where to save the projection weights
PROJ_PATH = "/home/chen_le/openset_detection/scripts/projection.pt"
os.makedirs(os.path.dirname(PROJ_PATH), exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Create projection (no seed needed; we just save whatever is initialized now)
proj = nn.Linear(C_MAX, C_OUT).to(device)
proj.eval()

# Save (set overwrite=True to replace existing)
overwrite = True
if os.path.exists(PROJ_PATH) and not overwrite:
    print(f"[SKIP] Projection already exists at {PROJ_PATH}. Set overwrite=True to replace.")
else:
    torch.save(proj.state_dict(), PROJ_PATH)
    print(f"[OK] Projection saved to: {PROJ_PATH}")

# Optional: print a simple checksum to recognize this projection
with torch.no_grad():
    wsum = torch.cat([p.flatten().cpu() for p in proj.parameters()]).sum().item()
print(f"[INFO] Projection checksum (sum of weights): {wsum:.6f}")

# [OK] Projection saved to: /home/chen_le/openset_detection/scripts/projection.pt
# [INFO] Projection checksum (sum of weights): 10.280164

[OK] Projection saved to: /home/chen_le/openset_detection/scripts/projection.pt
[INFO] Projection checksum (sum of weights): 1.197872
